# 07 — Structured Output with Pydantic

32 examples covering Guard.for_pydantic(), nested models, field constraints,
lists, enums, validators on fields, LLM structured output, REASK loops, and error aggregation.

**Installation:**
```bash
pip install guardrails-ai openai python-dotenv pydantic>=2.0
guardrails hub install hub://guardrails/valid_length
guardrails hub install hub://guardrails/valid_range
guardrails hub install hub://guardrails/regex_match
guardrails hub install hub://guardrails/toxic_language
```

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv('../.env')

import openai
from guardrails import Guard, OnFailAction
from guardrails.errors import ValidationError
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import List, Optional, Dict, Any, Union, Literal
from datetime import datetime
from enum import Enum

oai = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
MODEL = 'gpt-4o-mini'
print('Setup complete.')

In [ ]:
!guardrails hub install hub://guardrails/valid_length --quiet
!guardrails hub install hub://guardrails/valid_range --quiet
!guardrails hub install hub://guardrails/regex_match --quiet
!guardrails hub install hub://guardrails/toxic_language --quiet

## Examples 01–02: Simple Model and Guard.for_pydantic()

In [ ]:
# Example 01: Simple Pydantic model definition
class Person(BaseModel):
    name: str = Field(description='Full legal name')
    age: int = Field(description='Age in years')
    email: str = Field(description='Contact email')

print('Model schema:', Person.model_json_schema())

In [ ]:
# Example 02: Guard.for_pydantic() — wrap a Pydantic model in a Guard
class Person(BaseModel):
    name: str
    age: int

guard = Guard.for_pydantic(output_class=Person)
print('guard type:', type(guard).__name__)
print('json_schema:', guard.json_schema)

## Examples 03–04: Validate JSON String

In [ ]:
# Example 03: Valid JSON string parses to Pydantic model
class Product(BaseModel):
    name: str
    price: float
    in_stock: bool

guard = Guard.for_pydantic(output_class=Product)
outcome = guard.validate('{"name": "Widget", "price": 9.99, "in_stock": true}')
print('validated product:', outcome.validated_output)
print('type:', type(outcome.validated_output))

In [ ]:
# Example 04: Malformed JSON — missing required field
class Product(BaseModel):
    name: str
    price: float
    in_stock: bool

guard = Guard.for_pydantic(output_class=Product)
try:
    guard.validate('{"name": "Widget"}')  # missing price and in_stock
except ValidationError as e:
    print('FAIL - missing required fields:', str(e)[:120])

## Examples 05–06: Nested Models

In [ ]:
# Example 05: Nested Pydantic model definition
class Address(BaseModel):
    street: str
    city: str
    zip_code: str

class User(BaseModel):
    name: str
    address: Address

guard = Guard.for_pydantic(output_class=User)
valid_json = '{"name": "Alice", "address": {"street": "123 Main St", "city": "Portland", "zip_code": "97201"}}'
outcome = guard.validate(valid_json)
print('nested user:', outcome.validated_output)

In [ ]:
# Example 06: Missing city in nested address fails
class Address(BaseModel):
    street: str
    city: str

class User(BaseModel):
    name: str
    address: Address

guard = Guard.for_pydantic(output_class=User)
try:
    guard.validate('{"name": "Bob", "address": {"street": "456 Oak Ave"}}')
except ValidationError:
    print('FAIL - missing city in nested address')

## Examples 07–08: List Fields and Optional Fields

In [ ]:
# Example 07: List field — tags as list of strings
class Article(BaseModel):
    title: str
    tags: List[str]

guard = Guard.for_pydantic(output_class=Article)
outcome = guard.validate('{"title": "AI Guide", "tags": ["AI", "ML", "Python"]}')
print('article with tags:', outcome.validated_output)

In [ ]:
# Example 08: Optional fields — bio can be None
class Profile(BaseModel):
    username: str
    bio: Optional[str] = None
    website: Optional[str] = None

guard = Guard.for_pydantic(output_class=Profile)
# With optional fields
outcome1 = guard.validate('{"username": "alice"}')
outcome2 = guard.validate('{"username": "bob", "bio": "Developer", "website": "https://bob.dev"}')
print('minimal profile:', outcome1.validated_output)
print('full profile   :', outcome2.validated_output)

## Examples 09–10: Enum and Field Validators

In [ ]:
# Example 09: Enum/Literal field constrains allowed values
class Ticket(BaseModel):
    id: str
    priority: Literal['low', 'medium', 'high', 'critical']
    status: Literal['open', 'in_progress', 'closed']

guard = Guard.for_pydantic(output_class=Ticket)
outcome = guard.validate('{"id": "T-001", "priority": "high", "status": "open"}')
print('PASS - valid ticket:', outcome.validated_output)
try:
    guard.validate('{"id": "T-002", "priority": "urgent", "status": "open"}')
except ValidationError:
    print('FAIL - "urgent" not a valid priority')

In [ ]:
# Example 10: @field_validator alongside Guard
class Employee(BaseModel):
    name: str
    employee_id: str
    salary: float

    @field_validator('employee_id')
    @classmethod
    def validate_id_format(cls, v: str) -> str:
        if not v.startswith('EMP-'):
            raise ValueError('employee_id must start with EMP-')
        return v

guard = Guard.for_pydantic(output_class=Employee)
outcome = guard.validate('{"name": "Alice", "employee_id": "EMP-1234", "salary": 85000}')
print('PASS - valid employee:', outcome.validated_output)
try:
    guard.validate('{"name": "Bob", "employee_id": "12345", "salary": 70000}')
except ValidationError:
    print('FAIL - invalid employee_id format')

## Examples 11–12: Annotated Constraints and Custom Error Messages

In [ ]:
# Example 11: Annotated field constraints — age must be 0-120
from typing import Annotated

class Person(BaseModel):
    name: str
    age: Annotated[int, Field(ge=0, le=120)]
    score: Annotated[float, Field(ge=0.0, le=1.0)]

guard = Guard.for_pydantic(output_class=Person)
outcome = guard.validate('{"name": "Alice", "age": 30, "score": 0.85}')
print('PASS - valid constraints:', outcome.validated_output)
try:
    guard.validate('{"name": "Bob", "age": 150, "score": 0.5}')
except ValidationError:
    print('FAIL - age exceeds 120')

In [ ]:
# Example 12: Custom error messages via Field description
class ContactForm(BaseModel):
    name: str = Field(..., min_length=2, max_length=100, description='Full legal name, 2-100 characters')
    message: str = Field(..., min_length=10, max_length=1000, description='Support message, 10-1000 characters')

guard = Guard.for_pydantic(output_class=ContactForm)
try:
    guard.validate('{"name": "A", "message": "Hi"}')
except ValidationError as e:
    print('FAIL - field constraint violations:', str(e)[:150])

## Examples 13–14: LLM Structured Output and REASK

In [ ]:
# Example 13: LLM generates structured output matching Pydantic model
class BookRecommendation(BaseModel):
    title: str = Field(description='Book title')
    author: str = Field(description='Author full name')
    genre: str = Field(description='Book genre')
    rating: float = Field(description='Rating from 1.0 to 5.0')

guard = Guard.for_pydantic(output_class=BookRecommendation)
outcome = guard(
    oai.chat.completions.create,
    prompt='Recommend a science fiction book. Return JSON with title, author, genre, rating.',
    model=MODEL
)
print('LLM structured output:', outcome.validated_output)

In [ ]:
# Example 14: REASK when LLM forgets a required field
class WeatherReport(BaseModel):
    city: str
    temperature_celsius: float
    condition: str
    humidity_percent: int

guard = Guard.for_pydantic(output_class=WeatherReport)
outcome = guard(
    oai.chat.completions.create,
    prompt='Give me a weather report for London. Include city, temperature_celsius, condition, humidity_percent.',
    model=MODEL,
    num_reasks=2
)
print('weather report:', outcome.validated_output)

## Examples 15–16: FIX Wrong Type and Complex Schema

In [ ]:
# Example 15: FIX — string '25' coerced to int 25 for age field (Pydantic coercion)
class SimpleUser(BaseModel):
    name: str
    age: int  # Pydantic will coerce '25' -> 25

guard = Guard.for_pydantic(output_class=SimpleUser)
outcome = guard.validate('{"name": "Charlie", "age": "25"}')  # age as string
print('coerced output:', outcome.validated_output)
print('age type:', type(outcome.validated_output.age) if outcome.validated_output else None)

In [ ]:
# Example 16: Complex invoice schema with nested line items
class LineItem(BaseModel):
    description: str
    quantity: int
    unit_price: float
    total: float

class Invoice(BaseModel):
    invoice_number: str
    customer_name: str
    items: List[LineItem]
    subtotal: float
    tax_rate: float
    total_amount: float

guard = Guard.for_pydantic(output_class=Invoice)
outcome = guard(
    oai.chat.completions.create,
    prompt='Generate a sample invoice JSON for a software company with 2 line items.',
    model=MODEL
)
print('invoice generated:', outcome.validated_output)

## Examples 17–20: Union, Dict, Datetime, Root Validator

In [ ]:
# Example 17: Union types — result can be str or int
class FlexibleResult(BaseModel):
    query: str
    result: Union[str, int, float]

guard = Guard.for_pydantic(output_class=FlexibleResult)
outcome = guard.validate('{"query": "What is 2+2?", "result": 4}')
print('union int result:', outcome.validated_output)
outcome2 = guard.validate('{"query": "Color of sky?", "result": "blue"}')
print('union str result:', outcome2.validated_output)

In [ ]:
# Example 18: Dict field — metadata as free-form key-value pairs
class DataRecord(BaseModel):
    id: str
    data: Dict[str, Any]

guard = Guard.for_pydantic(output_class=DataRecord)
outcome = guard.validate('{"id": "rec-001", "data": {"source": "api", "version": 2, "tags": ["a", "b"]}}')
print('dict field result:', outcome.validated_output)

In [ ]:
# Example 19: Datetime field — ISO 8601 string parsed to datetime
class Event(BaseModel):
    name: str
    start_time: datetime
    end_time: datetime

guard = Guard.for_pydantic(output_class=Event)
outcome = guard.validate('{"name": "Team Meeting", "start_time": "2026-06-01T09:00:00", "end_time": "2026-06-01T10:00:00"}')
print('event:', outcome.validated_output)
if outcome.validated_output:
    print('start type:', type(outcome.validated_output.start_time))

In [ ]:
# Example 20: Root/model validator — cross-field: start_date < end_date
class DateRange(BaseModel):
    start_date: datetime
    end_date: datetime

    @model_validator(mode='after')
    def check_dates(self) -> 'DateRange':
        if self.start_date >= self.end_date:
            raise ValueError('start_date must be before end_date')
        return self

guard = Guard.for_pydantic(output_class=DateRange)
try:
    guard.validate('{"start_date": "2026-12-31T00:00:00", "end_date": "2026-01-01T00:00:00"}')
except ValidationError:
    print('FAIL - start_date is after end_date')

outcome = guard.validate('{"start_date": "2026-01-01T00:00:00", "end_date": "2026-12-31T00:00:00"}')
print('PASS - valid date range:', outcome.validation_passed)

## Examples 21–24: Guard Validators on Pydantic Fields

In [ ]:
# Example 21: Attach ToxicLanguage Guard validator to a Pydantic feedback field
from guardrails.hub import ToxicLanguage

class FeedbackForm(BaseModel):
    user_id: str
    feedback: str = Field(description='User feedback text')

guard = Guard.for_pydantic(output_class=FeedbackForm)
guard.use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('{"user_id": "u1", "feedback": "I hate this product, it is terrible!"}')
except ValidationError:
    print('FAIL - toxic feedback blocked')

In [ ]:
# Example 22: ValidLength on a Pydantic string field
from guardrails.hub import ValidLength

class Review(BaseModel):
    product_id: str
    review_text: str

guard = Guard.for_pydantic(output_class=Review)
guard.use(ValidLength(min=20, max=500, on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('{"product_id": "p1", "review_text": "Bad."}')
except ValidationError:
    print('FAIL - review text too short')

outcome = guard.validate('{"product_id": "p2", "review_text": "This product exceeded my expectations in every way."}')
print('PASS - review accepted:', outcome.validation_passed)

In [ ]:
# Example 23: Multiple validators on one field — ValidLength + ToxicLanguage on bio
from guardrails.hub import ValidLength, ToxicLanguage

class UserBio(BaseModel):
    username: str
    bio: str

guard = (
    Guard.for_pydantic(output_class=UserBio)
    .use(ValidLength(min=10, max=500, on_fail=OnFailAction.EXCEPTION))
    .use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
)
outcome = guard.validate('{"username": "alice", "bio": "Software engineer passionate about open source."}')
print('PASS - bio valid:', outcome.validation_passed)

In [ ]:
# Example 24: Field-level OnFailAction FILTER vs model-level EXCEPTION
from guardrails.hub import ToxicLanguage

class Post(BaseModel):
    title: str
    body: str

# FILTER: if validation fails on the body, filter that field out rather than raising
guard = Guard.for_pydantic(output_class=Post)
guard.use(ToxicLanguage(on_fail=OnFailAction.FILTER))
outcome = guard.validate('{"title": "My Post", "body": "Normal content here."}')
print('FILTER result with clean body:', outcome.validated_output)

## Examples 25–28: Advanced Pydantic Patterns

In [ ]:
# Example 25: Discriminated union — polymorphic response schema
class SuccessResponse(BaseModel):
    status: Literal['success']
    data: str

class ErrorResponse(BaseModel):
    status: Literal['error']
    message: str
    code: int

class APIResponse(BaseModel):
    response: Union[SuccessResponse, ErrorResponse]

guard = Guard.for_pydantic(output_class=APIResponse)
outcome = guard.validate('{"response": {"status": "success", "data": "User created"}}')
print('success discriminated union:', outcome.validated_output)

In [ ]:
# Example 26: OpenAI structured outputs mode with Guard
class ClassificationResult(BaseModel):
    category: Literal['sports', 'politics', 'technology', 'entertainment']
    confidence: float
    reasoning: str

guard = Guard.for_pydantic(output_class=ClassificationResult)
outcome = guard(
    oai.chat.completions.create,
    prompt='Classify this headline: "New AI model beats humans at chess". Return JSON.',
    model=MODEL,
    response_format={'type': 'json_object'}
)
print('classification:', outcome.validated_output)

In [ ]:
# Example 27: Top-level List[Model] response
class TodoItem(BaseModel):
    task: str
    priority: Literal['low', 'medium', 'high']
    done: bool = False

class TodoList(BaseModel):
    items: List[TodoItem]

guard = Guard.for_pydantic(output_class=TodoList)
outcome = guard(
    oai.chat.completions.create,
    prompt='Create a 3-item to-do list for learning Python. Return JSON with key "items" as a list.',
    model=MODEL
)
print('todo list:', outcome.validated_output)

In [ ]:
# Example 28: Pydantic v2 compatibility — model_dump(), model_json_schema()
class Config(BaseModel):
    host: str
    port: int
    debug: bool = False

guard = Guard.for_pydantic(output_class=Config)
outcome = guard.validate('{"host": "localhost", "port": 8080}')
if outcome.validated_output:
    print('model_dump:', outcome.validated_output.model_dump())
    print('model_json_schema:', Config.model_json_schema())

## Examples 29–32: Schema Introspection, REASK, Error Aggregation

In [ ]:
# Example 29: Guard.from_pydantic() vs Guard.for_pydantic() — usage comparison
class Item(BaseModel):
    name: str
    qty: int

# for_pydantic: primary way in newer versions
guard_for = Guard.for_pydantic(output_class=Item)
print('for_pydantic:', type(guard_for).__name__)
outcome = guard_for.validate('{"name": "Widget", "qty": 5}')
print('result:', outcome.validated_output)

In [ ]:
# Example 30: Schema introspection — inspect guard.json_schema
class SupportTicket(BaseModel):
    subject: str
    body: str
    priority: Literal['low', 'medium', 'high']

guard = Guard.for_pydantic(output_class=SupportTicket)
print('json_schema:')
print(json.dumps(guard.json_schema, indent=2))

In [ ]:
# Example 31: REASK with partial output — model returns 3 of 5 fields, reask fills remainder
class FullProfile(BaseModel):
    name: str
    email: str
    company: str
    role: str
    years_experience: int

guard = Guard.for_pydantic(output_class=FullProfile)
outcome = guard(
    oai.chat.completions.create,
    prompt='Create a sample developer profile JSON with all 5 fields: name, email, company, role, years_experience.',
    model=MODEL,
    num_reasks=2
)
print('full profile:', outcome.validated_output)

In [ ]:
# Example 32: Error aggregation — multiple field failures collected
class StrictRecord(BaseModel):
    name: str = Field(..., min_length=5)
    age: Annotated[int, Field(ge=18, le=65)]
    score: Annotated[float, Field(ge=0.0, le=100.0)]

guard = Guard.for_pydantic(output_class=StrictRecord)
try:
    # All three fields violate constraints
    guard.validate('{"name": "Hi", "age": 15, "score": 150.0}')
except ValidationError as e:
    print('Multiple validation errors collected:')
    print(str(e)[:300])